In [7]:
# Install the transformers library from HuggingFace
import subprocess
subprocess.run(["pip", "install", "transformers", "torch"])

import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

print("All libraries imported!")

# This tells you whether your Mac can use its chip to speed up training
# On newer MacBooks (M1/M2/M3) this may say 'mps' which is faster than cpu
print(f"Device available: {'mps' if torch.backends.mps.is_available() else 'cpu'}")

All libraries imported!
Device available: mps



[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [8]:
# Load the cleaned data
df = pd.read_csv('../data/cleaned_reviews.csv')

# DistilBERT needs labels as numbers not words
# convert  sentiment labels to integers
# negative = 0, neutral = 1, positive = 2
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Split into train and test sets
# the original review_text was used NOT clean_text
# because DistilBERT understands punctuation and capital letters
# and actually uses them as context clues
X_train, X_test, y_train, y_test = train_test_split(
    df['review_text'].fillna(''),  # fill any missing reviews with empty string
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']           # keep sentiment balance in both sets
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

Training samples: 3196
Testing samples:  799


## Tokenize the text

In [ ]:
# Load the DistilBERT tokeniser
# The tokeniser converts human text into numbers DistilBERT understands
# We use the same tokeniser that was used when DistilBERT was originally trained
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenise the training and test sets
# truncation=True  — cuts reviews longer than 512 words (DistilBERT's limit)
# padding=True     — adds zeros to short reviews so all inputs are the same length
# max_length=256   — we use 256 instead of 512 to speed up training.
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256
)

print("Tokenisation complete!")

Tokenisation complete!


## Create a pytorch dataset

In [10]:
# PyTorch needs the data in a special format called a Dataset
# Think of it as packaging your data into a format the model can open and read
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings        # the tokenised text
        self.labels = list(labels)        # the sentiment labels (0, 1, 2)

    def __len__(self):
        # returns how many reviews are in the dataset
        return len(self.labels)

    def __getitem__(self, idx):
        # returns one review at a time during training
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Package training and test data into datasets
train_dataset = ReviewDataset(train_encodings, y_train.values)
test_dataset  = ReviewDataset(test_encodings,  y_test.values)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size:  {len(test_dataset)}")

Training dataset size: 3196
Testing dataset size:  799


## Load Distilbert and set training settings

In [11]:
import subprocess
subprocess.run(["pip", "install", "accelerate>=1.1.0"])


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['pip', 'install', 'accelerate>=1.1.0'], returncode=0)

In [12]:


# Load the pretrained DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)

# Set the device — M1/M2/M3 Macs can use 'mps' for faster training
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model.to(device)

# Training settings
# These control how the model learns during training
training_args = TrainingArguments(
    output_dir='../models/distilbert',  # where to save the model
    num_train_epochs=3,                 # how many times to loop through all training data
    per_device_train_batch_size=16,     # how many reviews to process at once
    per_device_eval_batch_size=32,
    eval_strategy='epoch',        # evaluate after every epoch
    save_strategy='epoch',
    load_best_model_at_end=True,        # keep the best version of the model
    logging_dir='../models/logs',
    logging_steps=50,
    warmup_steps=100,                   # gradually increases learning rate at the start
    weight_decay=0.01                   # prevents the model from overfitting
)

print("Model and training settings ready!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model and training settings ready!


## Train the model

In [13]:
# Train the distilbert model
def compute_metrics(pred):
    # This function runs after each epoch to show us how well the model is doing
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    acc    = accuracy_score(labels, preds)
    return {'accuracy': acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting training... this will take a while, sit back and relax!")
trainer.train()
print("Training complete!")

/Users/emmanuelrockson/Desktop/ba-sentiment-analysis/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Starting training... this will take a while, sit back and relax!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.516116,0.631309,0.753442
2,0.402443,0.518330,0.782228
3,0.270057,0.574075,0.790989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/emmanuelrockson/Desktop/ba-sentiment-analysis/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/emmanuelrockson/Desktop/ba-sentiment-analysis/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


## Evaluate the model

In [14]:
# Get predictions on the test set
predictions = trainer.predict(test_dataset)
y_pred_bert = predictions.predictions.argmax(-1)

# Convert numbers back to labels for the report
label_names  = ['negative', 'neutral', 'positive']
y_test_names = [label_names[i] for i in y_test.values]
y_pred_names = [label_names[i] for i in y_pred_bert]

print("=" * 50)
print("DISTILBERT RESULTS")
print("=" * 50)
print(classification_report(y_test_names, y_pred_names))

# Final comparison of all three models
print("=" * 50)
print("FINAL MODEL COMPARISON")
print("=" * 50)
print(f"VADER Baseline:              66.0%")
print(f"TF-IDF + Logistic Regression: 74.8%")
print(f"DistilBERT:                  {accuracy_score(y_test_names, y_pred_names)*100:.1f}%")

/Users/emmanuelrockson/Desktop/ba-sentiment-analysis/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


DISTILBERT RESULTS
              precision    recall  f1-score   support

    negative       0.81      0.93      0.86       376
     neutral       0.42      0.31      0.35       144
    positive       0.89      0.84      0.86       279

    accuracy                           0.78       799
   macro avg       0.70      0.69      0.69       799
weighted avg       0.76      0.78      0.77       799

FINAL MODEL COMPARISON
VADER Baseline:              66.0%
TF-IDF + Logistic Regression: 74.8%
DistilBERT:                  78.2%


## Save the model

In [15]:
# Save the trained DistilBERT model and tokeniser
model.save_pretrained('../models/distilbert_final')
tokenizer.save_pretrained('../models/distilbert_final')

print("DistilBERT model and tokeniser saved to models/distilbert_final/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT model and tokeniser saved to models/distilbert_final/
